### Исходные данные

In [3]:
import numpy as np
from numba import njit
import pandas as pd
import time

package_sizes = np.array([25.0, 200.0, 100.0])
commissions = np.array([0.04, 0.07, 0.05])
min_investments = np.array([30.0, 150.0, 100.0])

# Вероятности по этапам 
probabilities = np.array([
    [0.60, 0.30, 0.10],
    [0.30, 0.20, 0.50],
    [0.40, 0.40, 0.20],
])

# Коэффициенты
price_changes = np.array([
    [[1.20, 1.10, 1.07], [1.05, 1.02, 1.03], [0.80, 0.95, 1.00]],
    [[1.40, 1.15, 1.01], [1.05, 1.00, 1.00], [0.60, 0.90, 1.00]],
    [[1.15, 1.12, 1.05], [1.05, 1.01, 1.01], [0.70, 0.94, 1.00]],
])

initial_portfolio = np.array([100.0, 800.0, 400.0, 600.0])
eps = 1e-7  

average_last = probabilities[2] @ price_changes[2]

print('Начальный капитал:', initial_portfolio.sum())
print('Средние коэффициенты третьего этапа:', average_last)


Начальный капитал: 1900.0
Средние коэффициенты третьего этапа: [1.02  1.04  1.024]


### Покупка, продажа и проверка ограничений



In [4]:
@njit
def total_capital(portfolio):
    return portfolio[0] + portfolio[1] + portfolio[2] + portfolio[3]


# Покупка и продажа с учётом комиссий
@njit
def make_trades(portfolio, trades):
    updated = portfolio.copy()
    for asset in range(3):
        amount = package_sizes[asset] * trades[asset]
        fee = commissions[asset] * abs(amount)
        updated[asset] += amount
        updated[3] -= amount + fee
    return updated


# Проверка ограничений на сделки
@njit
def trades_allowed(portfolio, trades):
    updated = make_trades(portfolio, trades)
    for asset in range(3):
        if updated[asset] < min_investments[asset] - eps:
            return False
    return updated[3] >= -eps


# Изменение стоимости активов
@njit
def market_change(portfolio_after_trades, stage, situation):
    updated = portfolio_after_trades.copy()
    for asset in range(3):
        updated[asset] *= price_changes[stage, situation, asset]
    return updated


# Минимальное число пакетов для докупки
@njit
def minimum_packages(portfolio, asset):
    difference = min_investments[asset] - portfolio[asset]
    return int(np.ceil((difference - eps) / package_sizes[asset]))


# Ожидаемый капитал после последнего этапа
@njit
def expected_last_capital(portfolio_after_trades):
    result = portfolio_after_trades[3]
    for asset in range(3):
        result += portfolio_after_trades[asset] * average_last[asset]
    return result


# Проверка на примере сделки
example_trades = np.array([2, -1, 0])
example_result = make_trades(initial_portfolio, example_trades)

print('После сделок:', example_result)
print('Сделка допустима:', trades_allowed(initial_portfolio, example_trades))

После сделок: [150. 600. 400. 734.]
Сделка допустима: True


### Границы перебора

In [5]:
@njit
def trade_limits(portfolio):
    lower = np.zeros(3, dtype=np.int64)
    upper = np.zeros(3, dtype=np.int64)
    money_total = total_capital(portfolio)

    for asset in range(3):
        # нижняя граница
        lower[asset] = minimum_packages(portfolio, asset)

        # верхняя граница
        other_min = 0.0
        for other in range(3):
            if other != asset:
                other_min += min_investments[other]

        available = money_total - other_min - min_investments[asset] + portfolio[asset]
        upper[asset] = int(np.floor((available + eps) / package_sizes[asset]))
    return lower, upper


limits_min, limits_max = trade_limits(initial_portfolio)
print('Нижние границы:', limits_min)
print('Верхние границы:', limits_max)


Нижние границы: [-2 -3 -3]
Верхние границы: [68 12 20]


### Третий этап — начало обратного расчёта

Ожидаемый конечный капитал после сделок

In [6]:
@njit
def bellman_stage3(portfolio):
    # Минимальные обязательные покупки
    required = np.zeros(3, dtype=np.int64)
    money_needed = 0.0
    for asset in range(3):
        required[asset] = max(0, minimum_packages(portfolio, asset))
        money_needed += package_sizes[asset] * (1 + commissions[asset]) * required[asset]

    if portfolio[3] + eps >= money_needed:
        final_portfolio = make_trades(portfolio, required)
        return expected_last_capital(final_portfolio), required

    deficit = money_needed - portfolio[3]
    sell_limits = np.zeros(3, dtype=np.int64)
    for asset in range(3):
        if required[asset] == 0:
            possible = max(0, int(np.floor(
                (portfolio[asset] - min_investments[asset] + eps) / package_sizes[asset]
            )))
            enough = int(np.ceil(deficit / (package_sizes[asset] * (1 - commissions[asset])))) + 1
            sell_limits[asset] = min(possible, enough)

    best_value = -np.inf
    best_trades = np.zeros(3, dtype=np.int64)
    for sold1 in range(sell_limits[0] + 1):
        for sold2 in range(sell_limits[1] + 1):
            for sold3 in range(sell_limits[2] + 1):
                trades = np.array((required[0] - sold1,
                                   required[1] - sold2,
                                   required[2] - sold3), dtype=np.int64)
                if not trades_allowed(portfolio, trades):
                    continue
                value = expected_last_capital(make_trades(portfolio, trades))
                if value > best_value + 1e-9:
                    best_value = value
                    best_trades = trades
    return best_value, best_trades


### рекурсия Беллмана

Для каждого допустимого управления смотрим три возможных исхода и берём средний результат с учётом вероятностей. 

In [7]:
@njit
def bellman_stage2(portfolio):
    lower, upper = trade_limits(portfolio)
    best_value = -np.inf
    best_trades = np.zeros(3, dtype=np.int64)

    # Перебор количества пакетов каждого актива
    for packages1 in range(lower[0], upper[0] + 1):
        for packages2 in range(lower[1], upper[1] + 1):
            for packages3 in range(lower[2], upper[2] + 1):
                trades = np.array((packages1, packages2, packages3), dtype=np.int64)
                if not trades_allowed(portfolio, trades):
                    continue

                after_trades = make_trades(portfolio, trades)
                average_result = 0.0

                # Три рыночные ситуации второго этапа
                for situation in range(3):
                    next_portfolio = market_change(after_trades, 1, situation)
                    stage3_value, _ = bellman_stage3(next_portfolio)
                    if not np.isfinite(stage3_value):
                        average_result = -np.inf
                        break
                    average_result += probabilities[1, situation] * stage3_value

                if average_result > best_value + 1e-9:
                    best_value = average_result
                    best_trades = trades

    return best_value, best_trades


@njit
def bellman_stage1(portfolio):
    lower, upper = trade_limits(portfolio)
    best_value = -np.inf
    best_trades = np.zeros(3, dtype=np.int64)

    for packages1 in range(lower[0], upper[0] + 1):
        for packages2 in range(lower[1], upper[1] + 1):
            for packages3 in range(lower[2], upper[2] + 1):
                trades = np.array((packages1, packages2, packages3), dtype=np.int64)
                if not trades_allowed(portfolio, trades):
                    continue

                after_trades = make_trades(portfolio, trades)
                average_result = 0.0

                # Три рыночные ситуации первого этапа
                for situation in range(3):
                    next_portfolio = market_change(after_trades, 0, situation)
                    stage2_value, _ = bellman_stage2(next_portfolio)
                    if not np.isfinite(stage2_value):
                        average_result = -np.inf
                        break
                    average_result += probabilities[0, situation] * stage2_value

                if average_result > best_value + 1e-9:
                    best_value = average_result
                    best_trades = trades

    return best_value, best_trades


### Оптимальное первое управление


In [8]:
start_time = time.perf_counter()
optimal_capital, first_trades = bellman_stage1(initial_portfolio)
calculation_time = time.perf_counter() - start_time

print('Первое управление (ЦБ1, ЦБ2, депозит):', tuple(first_trades))
print(f'Ожидаемый конечный капитал: {optimal_capital:.6f} д.е.')
print(f'Ожидаемый доход: {optimal_capital - total_capital(initial_portfolio):.6f} д.е.')
print(f'Время расчёта с компиляцией: {calculation_time:.2f} с')


Первое управление (ЦБ1, ЦБ2, депозит): (np.int64(23), np.int64(0), np.int64(0))
Ожидаемый конечный капитал: 2033.988884 д.е.
Ожидаемый доход: 133.988884 д.е.
Время расчёта с компиляцией: 10.52 с


### Прямой проход — восстановление стратегии

После обратного расчёта идём вперёд: сначала делаем первую найденную сделку, затем для каждого из трёх исходов считаем действия второго этапа, после этого — третьего. Получаем **9 ветвей управлений** и **27 конечных исходов**.


In [9]:
situation_names = ['Благоприятная', 'Нейтральная', 'Негативная']
strategy_rows = []
end_rows = []

portfolio_after_first = make_trades(initial_portfolio, first_trades)
assert trades_allowed(initial_portfolio, first_trades)

for first_situation in range(3):
    portfolio2 = market_change(portfolio_after_first, 0, first_situation)
    value2, trades2 = bellman_stage2(portfolio2)
    assert trades_allowed(portfolio2, trades2)
    portfolio_after_second = make_trades(portfolio2, trades2)

    for second_situation in range(3):
        portfolio3 = market_change(portfolio_after_second, 1, second_situation)
        value3, trades3 = bellman_stage3(portfolio3)
        assert np.isfinite(value3) and trades_allowed(portfolio3, trades3)
        portfolio_after_third = make_trades(portfolio3, trades3)

        strategy_rows.append({
            'Исход 1': situation_names[first_situation],
            'Исход 2': situation_names[second_situation],
            'Управление 2': tuple(int(x) for x in trades2),
            'Управление 3': tuple(int(x) for x in trades3),
            'W3': round(value3, 6),
        })

        for third_situation in range(3):
            final_portfolio = market_change(portfolio_after_third, 2, third_situation)
            chance = (probabilities[0, first_situation] *
                      probabilities[1, second_situation] *
                      probabilities[2, third_situation])
            end_rows.append({
                'Исход 1': situation_names[first_situation],
                'Исход 2': situation_names[second_situation],
                'Исход 3': situation_names[third_situation],
                'Вероятность': chance,
                'Капитал': total_capital(final_portfolio),
            })

strategy_table = pd.DataFrame(strategy_rows)
end_table = pd.DataFrame(end_rows)
print('Первое управление:', tuple(first_trades))
display(strategy_table)


Первое управление: (np.int64(23), np.int64(0), np.int64(0))


,Исход 1,Исход 2,Управление 2,Управление 3,W3
0,Благоприятная,Благоприятная,"(-31, 0, 0)","(0, 0, 0)",2291.11472
1,Благоприятная,Нейтральная,"(-31, 0, 0)","(0, 0, 0)",2136.95700
2,Благоприятная,Негативная,"(-31, 0, 0)","(1, 0, 0)",2028.87200
3,Нейтральная,Благоприятная,"(-27, 0, 0)","(0, 0, 0)",2100.23788
4,Нейтральная,Нейтральная,"(-27, 0, 0)","(0, 0, 0)",1956.67425
5,Нейтральная,Негативная,"(-27, 0, 0)","(1, 0, 0)",1855.81900
6,Негативная,Благоприятная,"(-20, 0, 0)","(0, 0, 0)",1861.77600
7,Негативная,Нейтральная,"(-20, 0, 0)","(0, 0, 0)",1724.84000
8,Негативная,Негативная,"(-20, 0, 0)","(1, 0, 0)",1626.94000


### Проверка

Умножаем капитал в каждой конечной ветви на вероятность этой ветви. Сумма должна совпадать со значением Беллмана, найденным в обратном расчёте.

In [10]:
expected_from_branches = (end_table['Вероятность'] * end_table['Капитал']).sum()
print('Сумма вероятностей:', end_table['Вероятность'].sum())
print('По Беллману:', round(optimal_capital, 6))
print('По всем ветвям:', round(expected_from_branches, 6))
assert np.isclose(expected_from_branches, optimal_capital, atol=1e-6)
display(end_table.round(6))


Сумма вероятностей: 1.0
По Беллману: 2033.988884
По всем ветвям: 2033.988884


,Исход 1,Исход 2,Исход 3,Вероятность,Капитал
0,Благоприятная,Благоприятная,Благоприятная,0.072,2389.684000
1,Благоприятная,Благоприятная,Нейтральная,0.072,2256.172800
2,Благоприятная,Благоприятная,Негативная,0.036,2163.860000
3,Благоприятная,Нейтральная,Благоприятная,0.048,2223.262500
4,Благоприятная,Нейтральная,Нейтральная,0.048,2105.667500
5,Благоприятная,Нейтральная,Негативная,0.024,2026.925000
6,Благоприятная,Негативная,Благоприятная,0.120,2109.340000
7,Благоприятная,Негативная,Нейтральная,0.120,2000.500000
8,Благоприятная,Негативная,Негативная,0.060,1924.680000
9,Нейтральная,Благоприятная,Благоприятная,0.036,2192.271500
